### Transform Refunds Data

1. Extract specific portion of the string from refund_reason using split function  
2. Extract specific portion of the string from refund_reason using regexp_extract function  
3. Extract date and time from the refund_timestamp  
4. Write transformed data to the Silver schema in hive metastore

##### 1. Extract specific portion of the string from refund_reason using split function

In [0]:
SELECT 
refund_id,
payment_id,
refund_timestamp,
refund_amount,
SPLIT(refund_reason, ':')[0] AS refund_reason,
SPLIT(refund_reason, ':')[1] AS refund_source 
FROM hive_metastore.bronze.refunds;

##### 2. Extract specific portion of the string from refund_reason using regexp_extract function

In [0]:
SELECT refund_id,
       payment_id,
       CAST(date_format(refund_timestamp, 'yyyy-MM-dd') AS DATE) AS refund_date,
       refund_amount,
       regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
       regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) AS refund_source
FROM hive_metastore.bronze.refunds;

##### 3. Write transformed data to the Silver schema in hive metastore

In [0]:
CREATE SCHEMA hive_metastore.silver;

In [0]:
CREATE TABLE hive_metastore.silver.refunds
AS
SELECT refund_id,
       payment_id,
       CAST(date_format(refund_timestamp, 'yyyy-MM-dd') AS DATE) AS refund_date,
       refund_amount,
       regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
       regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) AS refund_source
FROM hive_metastore.bronze.refunds;

In [0]:
SELECT * FROM hive_metastore.silver.refunds;

In [0]:
DESCRIBE EXTENDED hive_metastore.silver.refunds;

If we delete the workspace then table/data in the root location dbfs:/user/hive/metastore will also be deleted